# CABG readmission: Table 1 + Random Forest (row2 imputed)

This notebook:
- loads `outcomes/s1/cleaned_dataset_row2_imputed.csv`
- creates Table 1 (baseline characteristics by readmission status) with p-values
- trains / evaluates a Random Forest model for readmission
- saves all outputs into `outcomes/s2/`

Notes:
- `Readmission` is expected to be coded 0/1 (0 = No, 1 = Yes).
- The notebook is designed to run when placed inside the `scripts/` folder of your project.


### Chunk 1 — Imports + Paths + Load

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy import stats

# ----------------------------
# project paths (robust)
# ----------------------------
def find_project_root(start: Path) -> Path:
    """Walk upward until we find a folder that contains both 'datasets' and 'outcomes'."""
    cur = start.resolve()
    for _ in range(10):
        if (cur / "datasets").exists() and (cur / "outcomes").exists():
            return cur
        cur = cur.parent
    return start.resolve()

project_root = find_project_root(Path.cwd())
in_fp = project_root / "outcomes" / "s1" / "cleaned_dataset_row2_imputed.csv"
out_dir = project_root / "outcomes" / "s2"
out_dir.mkdir(parents=True, exist_ok=True)

print("project_root:", project_root)
print("input:", in_fp)
print("output dir:", out_dir)
assert in_fp.exists(), f"Input file not found: {in_fp}"

# ----------------------------
# load data
# ----------------------------
df = pd.read_csv(in_fp)
print("shape:", df.shape)
print("columns(raw):", list(df.columns))


### Chunk 2 — Rename to snake_case + Pretty map + Sanity checks (Male=1)

In [ ]:
def to_snake(s: str) -> str:
    s = str(s).strip()
    s = s.replace("%", " pct ")
    s = re.sub(r"[\/\-]+", " ", s)
    s = re.sub(r"[^0-9a-zA-Z ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    s = s.replace(" ", "_")
    return s

# rename columns to snake_case
orig_cols = df.columns.tolist()
df.columns = [to_snake(c) for c in orig_cols]
print("columns(snake):", list(df.columns))

# Pretty-name map (snake_case keys)
PRETTY_MAP_RAW = {
    "albumin":"Albumin",
    "bilirubin":"Bilirubin",
    "creatinine":"Creatinine",
    "diabetes":"Diabetes",
    "endocarditis":"Endocarditis",
    "gender":"Gender (Male)",
    "hematocrit":"Hematocrit",
    "hemoglobin":"Hemoglobin",
    "home_oxygen":"Home Oxygen",
    "hypertension":"Hypertension",
    "meld_score":"MELD Score",
    "mediastinal_radiation":"Mediastinal Radiation",
    "platelets":"Platelets",
    "pneumonia":"Pneumonia",
    "race":"Race",
    "sleep_apnea":"Sleep Apnea",
    "syncope":"Syncope",
    "a1c_level":"Hemoglobin A1c",
    "alcohol_use":"Alcohol use",
    "bmi":"Body mass index",
    "cld":"Chronic lung disease",
    "cvd":"Cerebrovascular disease",
    "hemo_data_ef":"Ejection Fraction",
    "illicit_drug_use":"Illicit drug use",
    "immunocompromise":"Immunocompromised",
    "immunocompromised":"Immunocompromised",
    "liver_disease":"Liver disease",
    "num_dis_vessels":"Number of Diseased Vessels",
    "pad":"Peripheral artery disease",
    "patient_age":"Age",
    "prev_cardiac_intervent":"Previous cardiac intervention",
    "previous_pci":"Previous cardiac intervention",
    "prior_mi":"Prior myocardial Infarction",
    "status":"Admission status",
    "tobacco_use":"Tobacco use",
    "wbc_count":"White blood cell",
    "readmission":"Readmission",
    "inr":"INR",
    "ejection_fraction":"Ejection Fraction",
}

# sanity checks
assert "readmission" in df.columns, "readmission column missing after renaming."
u = sorted(df["readmission"].dropna().unique().tolist())
print("readmission unique:", u)
assert set(u).issubset({0, 1}), f"readmission must be 0/1, got {u}"

# gender is male=1 confirmed by you; ensure numeric
if "gender" in df.columns:
    df["gender"] = pd.to_numeric(df["gender"], errors="coerce")
    ug = sorted(df["gender"].dropna().unique().tolist())
    print("gender unique:", ug, "(expect 0/1, male=1)")
    assert set(ug).issubset({0, 1}), f"gender must be 0/1, got {ug}"

# convert other columns to numeric if possible (no-op if already numeric)
for c in df.columns:
    if c in ["s_no"]:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        continue
    df[c] = pd.to_numeric(df[c], errors="ignore")


### Chunk 3 — Make Table 1 (block style + correct binary display)

In [ ]:
def format_mean_sd(x: pd.Series) -> str:
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return "NA"
    return f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

def format_n_pct_event(x: pd.Series, event_value: int = 1) -> str:
    x = pd.to_numeric(x, errors="coerce").dropna()
    n = len(x)
    if n == 0:
        return "0 (0.0%)"
    n_event = int((x == event_value).sum())
    pct = 100.0 * n_event / n
    return f"{n_event} ({pct:.1f}%)"

def p_value_continuous(x0: pd.Series, x1: pd.Series) -> float:
    x0 = pd.to_numeric(x0, errors="coerce").dropna()
    x1 = pd.to_numeric(x1, errors="coerce").dropna()
    if (len(x0) < 2) or (len(x1) < 2):
        return np.nan
    return stats.ttest_ind(x0, x1, equal_var=False, nan_policy="omit").pvalue

def p_value_categorical(x: pd.Series, y: pd.Series) -> float:
    tmp = pd.crosstab(x, y, dropna=True)
    if tmp.shape[0] == 0 or tmp.shape[1] == 0:
        return np.nan
    chi2, p, dof, exp = stats.chi2_contingency(tmp.values)
    return p

def fmt_p_display(p: float) -> str:
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}".lstrip("0")

def make_table1(df: pd.DataFrame, group_col: str,
                vars_continuous: list, vars_binary: list, vars_categorical: list) -> pd.DataFrame:
    g = df[group_col]
    df0 = df[g == 0].copy()
    df1 = df[g == 1].copy()

    rows = []

    # -------- continuous blocks (each variable is its own block) --------
    for v in vars_continuous:
        if v not in df.columns:
            continue
        p = p_value_continuous(df0[v], df1[v])
        label = PRETTY_MAP_RAW.get(v, v)
        rows.append({
            "Variable": label,
            "Readmission: No": format_mean_sd(df0[v]),
            "Readmission: Yes": format_mean_sd(df1[v]),
            "P value": fmt_p_display(p),
            "P value (num)": p
        })

    # -------- binary blocks (each variable is its own block) --------
    # rule: 1 means "event/Yes"; for Gender we also treat 1 as Male (per you)
    for v in vars_binary:
        if v not in df.columns:
            continue
        p = p_value_categorical(df[v], g)
        label = PRETTY_MAP_RAW.get(v, v)
        rows.append({
            "Variable": label,
            "Readmission: No": format_n_pct_event(df0[v], event_value=1),
            "Readmission: Yes": format_n_pct_event(df1[v], event_value=1),
            "P value": fmt_p_display(p),
            "P value (num)": p
        })

    # -------- categorical blocks (header + indented subrows) --------
    for v in vars_categorical:
        if v not in df.columns:
            continue

        label = PRETTY_MAP_RAW.get(v, v)
        p = p_value_categorical(df[v], g)

        # header row for the block (holds p-value)
        rows.append({
            "Variable": label,
            "Readmission: No": "",
            "Readmission: Yes": "",
            "P value": fmt_p_display(p),
            "P value (num)": p
        })

        cats = sorted(pd.Series(df[v].dropna().unique()).tolist())
        for cat in cats:
            n0 = int((df0[v] == cat).sum()); d0 = int(df0[v].notna().sum())
            n1 = int((df1[v] == cat).sum()); d1 = int(df1[v].notna().sum())
            rows.append({
                "Variable": f"  {cat}",  # indented subrow
                "Readmission: No": f"{n0} ({(100*n0/d0 if d0 else 0):.1f}%)",
                "Readmission: Yes": f"{n1} ({(100*n1/d1 if d1 else 0):.1f}%)",
                "P value": "",
                "P value (num)": np.nan
            })

    out = pd.DataFrame(rows)
    return out

# ----------------------------
# variables to include (snake_case names)
# ----------------------------
vars_cont = [
    "patient_age","bmi","wbc_count","hematocrit","platelets","albumin",
    "a1c_level","creatinine","bilirubin","inr","meld_score","ejection_fraction"
]

vars_bin = [
    "gender","diabetes","hypertension","endocarditis","sleep_apnea",
    "liver_disease","immunocompromised","pad","syncope","cvd","prior_mi",
    "tobacco_use"
]

# These might be multi-level in your cleaned file; auto-correct below
maybe_multi = ["pneumonia","illicit_drug_use","alcohol_use","status"]
vars_bin += [v for v in maybe_multi if v in df.columns]

vars_cat = ["race","cld","num_dis_vessels"]

# auto-fix: if any "binary" var isn't {0,1}, treat it as categorical
auto_cat = []
auto_bin = []
for v in vars_bin:
    if v not in df.columns:
        continue
    uu = sorted(pd.Series(df[v].dropna().unique()).tolist())
    if len(uu) <= 2 and set(uu).issubset({0, 1}):
        auto_bin.append(v)
    else:
        auto_cat.append(v)

vars_bin = auto_bin
vars_cat = list(dict.fromkeys([v for v in vars_cat if v in df.columns] + auto_cat))

print("Table 1 binary vars:", vars_bin)
print("Table 1 categorical vars:", vars_cat)

table1 = make_table1(df, "readmission", vars_cont, vars_bin, vars_cat)
table1.head(15)


### Chunk 4 — Filter blocks by p<=0.05 + DOCX export (uses table1, not t1)

In [ ]:
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.shared import OxmlElement, qn

def set_cell_margins(cell, top=80, start=80, bottom=80, end=80):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    tcMar = OxmlElement('w:tcMar')
    for m, v in (('top', top), ('start', start), ('bottom', bottom), ('end', end)):
        node = OxmlElement(f'w:{m}')
        node.set(qn('w:w'), str(v))
        node.set(qn('w:type'), 'dxa')
        tcMar.append(node)
    tcPr.append(tcMar)

def build_table1_docx(df_doc: pd.DataFrame, out_path: Path,
                      title: str = "Table 1. Baseline characteristics by readmission status (p ≤ 0.05)"):

    # drop helper numeric col if present
    if "P value (num)" in df_doc.columns:
        df_doc = df_doc.drop(columns=["P value (num)"])

    doc = Document()

    p = doc.add_paragraph(title)
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT

    table = doc.add_table(rows=1, cols=df_doc.shape[1])
    table.style = "Table Grid"

    hdr_cells = table.rows[0].cells
    for j, col in enumerate(df_doc.columns):
        hdr_cells[j].text = str(col)
        set_cell_margins(hdr_cells[j])

    for i in range(df_doc.shape[0]):
        row_cells = table.add_row().cells
        for j, col in enumerate(df_doc.columns):
            val = "" if pd.isna(df_doc.iloc[i, j]) else str(df_doc.iloc[i, j])
            row_cells[j].text = val
            set_cell_margins(row_cells[j])
            for par in row_cells[j].paragraphs:
                par.alignment = WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.RIGHT

        v = str(df_doc.iloc[i, 0])
        if v.startswith("  "):
            row_cells[0].text = v.strip()
            for par in row_cells[0].paragraphs:
                par.paragraph_format.left_indent = Inches(0.2)

    style = doc.styles["Normal"]
    style.font.name = "Times New Roman"
    style.font.size = Pt(10)

    doc.save(out_path)

def filter_blocks_by_pvalue(t1: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    Keep blocks where the header row has p <= alpha.
    Block definition:
      - header row: Variable does NOT start with two spaces
      - subrows: Variable starts with two spaces
    """
    df = t1.copy()

    # must have numeric p to filter
    if "P value (num)" not in df.columns:
        df["P value (num)"] = pd.to_numeric(df.get("P value", ""), errors="coerce")

    pnum = pd.to_numeric(df["P value (num)"], errors="coerce")

    keep_rows = []
    keep_current_block = True

    for i in range(len(df)):
        var = str(df.iloc[i, 0])
        is_header = not var.startswith("  ")

        if is_header:
            pi = pnum.iloc[i]
            if np.isnan(pi):
                keep_current_block = True
            else:
                keep_current_block = (pi <= alpha)

        if keep_current_block:
            keep_rows.append(i)

    df_out = df.iloc[keep_rows].copy()

    # safety: keep display p only (it is already formatted)
    return df_out

table1_sig = filter_blocks_by_pvalue(table1, alpha=0.05)

docx_path = out_dir / "Table1_descriptive_p_le_0p05.docx"
csv_path  = out_dir / "Table1_descriptive_p_le_0p05.csv"

build_table1_docx(table1_sig, docx_path)
table1_sig.drop(columns=["P value (num)"], errors="ignore").to_csv(csv_path, index=False)

print("Saved:", docx_path)
print("Saved:", csv_path)
